In [1]:
# Parts of this notebook are adapted from the code accompanying Moon and Lazar (2023).

In [21]:
import sys
import os
sys.path.append(os.path.abspath(".."))
sys.path.append(os.path.abspath("../../source"))
from TDA_Testing import *
import json
import pandas as pd
import pickle
def savepkl(obj, path):
    with open(path, 'wb') as f:
        pickle.dump(obj, f)
def loadpkl(path):
    with open(path, 'rb') as f:
        return pickle.load(f)

# Data Load

In [22]:
# import PD - json, each pds are saved as unnamed array

with open("../../data/json/npc=50_nset=100onepd.json") as f:
    onepd = json.load(f)
with open("../../data/json/npc=50_nset=100twopd.json") as f:
    twopd = json.load(f)

# convert json (saved as unnamed array) to python list
onepdlist=[]
for ii in range(len(onepd)):
    onepdsublist = []
    for jj in range(len(onepd[0])):
        onepddat = np.array(onepd[ii][jj])
        onepdmat = np.transpose( np.resize(onepddat, (3,int(len(onepddat)/3)) ) )
        onepddim1 = onepdmat[onepdmat[:,0]==1,1:]
        onepdsublist.append(onepddim1)
    onepdlist.append(onepdsublist)
    
twopdlist=[]
for ii in range(len(twopd)):
    twopdsublist = []
    for jj in range(len(twopd[0])):
        twopddat = np.array(twopd[ii][jj])
        twopdmat = np.transpose( np.resize(twopddat, (3,int(len(twopddat)/3)) ) )
        twopddim1 = twopdmat[twopdmat[:,0]==1,1:]
        twopdsublist.append(twopddim1)
    twopdlist.append(twopdsublist)

# Simulation with different noise

In [27]:
# simulation parameters
npc=10
nset=100
sig=[0.05,0.10,0.15,0.20]
nsig=len(sig)

## Aggregation test

In [19]:
#linear_weight
random.seed(42)
func_weight = function_weight("Poly", poly_order=1)
Agg_linear_opt=np.zeros((nsig,nset))
for n in range(nsig):
    print(n)
    print(np.sum(Agg_linear_opt==1,axis=1)/nset)
    for jj in range(nset):
        X = onepdlist[n][npc*jj:npc*(jj+1)]
        Y = twopdlist[n][npc*jj:npc*(jj+1)]
        test_result = Aggtest(X,Y,optimal_bandwidths=True,weight_function=func_weight,Rff_approx=True)
        Agg_linear_opt[n,jj] = test_result

0
[0. 0. 0. 0.]
1
[1. 0. 0. 0.]
2
[1.   0.54 0.   0.  ]
3
[1.   0.54 0.12 0.  ]


In [20]:
print(np.sum(Agg_linear_opt==1,axis=1)/nset) 
savepkl(np.sum(Agg_linear_opt==1,axis=1)/nset, '../../results/simulation_results/Circle_results/Circles_Agg_linear_opt_diffnoise.pkl')

[1.   0.54 0.12 0.09]


In [15]:
#constant_weight
random.seed(42)
func_weight = function_weight("constant")
Agg_constant_opt=np.zeros((nsig,nset))
for n in range(nsig):
    print(n)
    print(np.sum(Agg_constant_opt==1,axis=1)/nset)
    for jj in range(nset):
        X = onepdlist[n][npc*jj:npc*(jj+1)]
        Y = twopdlist[n][npc*jj:npc*(jj+1)]
        test_result = Aggtest(X,Y,optimal_bandwidths=True,weight_function=func_weight,Rff_approx=True)
        Agg_constant_opt[n,jj] = test_result

0
[0. 0. 0. 0.]
1
[1. 0. 0. 0.]
2
[1.   0.71 0.   0.  ]
3
[1.   0.71 0.26 0.  ]


In [16]:
print(np.sum(Agg_constant_opt==1,axis=1)/nset) 
savepkl(np.sum(Agg_constant_opt==1,axis=1)/nset, '../../results/simulation_results/Circle_results/Circles_Agg_constant_opt_diffnoise.pkl')

[1.   0.71 0.26 0.13]


In [24]:
#arctan_weight
random.seed(42)
func_weight = function_weight("arctan")
Agg_arctan_opt=np.zeros((nsig,nset))
for n in range(nsig):
    print(n)
    print(np.sum(Agg_arctan_opt==1,axis=1)/nset)
    for jj in range(nset):
        X = onepdlist[n][npc*jj:npc*(jj+1)]
        Y = twopdlist[n][npc*jj:npc*(jj+1)]
        test_result = Aggtest(X,Y,optimal_bandwidths=True,weight_function=func_weight,Rff_approx=True)
        Agg_arctan_opt[n,jj] = test_result

0
[0. 0. 0. 0.]
1
[1. 0. 0. 0.]
2
[1.  0.5 0.  0. ]
3
[1.   0.5  0.14 0.  ]


In [25]:
print(np.sum(Agg_arctan_opt==1,axis=1)/nset) 
savepkl(np.sum(Agg_arctan_opt==1,axis=1)/nset, '../../results/simulation_results/Circle_results/Circles_Agg_arctan_opt_diffnoise.pkl')

[1.   0.5  0.14 0.08]


## PD test 

In [29]:
random.seed(42)
Perm=np.zeros((nsig,nset))
for n in range(nsig):
    print(n)
    print(np.sum(Perm<0.05,axis=1)/nset)
    for jj in range(nset):
        X = onepdlist[n][npc*jj:npc*(jj+1)]
        Y = twopdlist[n][npc*jj:npc*(jj+1)]
        T_obs, T_perm, p_val = permutation_test(X, Y, num_permutations=1000)
        Perm[n,jj] = p_val

0
[1. 1. 1. 1.]
1
[1. 1. 1. 1.]
2
[1.   0.56 1.   1.  ]
3
[1.   0.56 0.25 1.  ]


In [30]:
savepkl(np.sum(Perm<0.05,axis=1)/nset, '../../results/simulation_results/Circle_results/Circles_PD_result_diffnoise.pkl')
print(np.sum(Perm<0.05,axis=1)/nset)

[1.   0.56 0.25 0.1 ]


## PL test

In [31]:
# convert json (saved as unnamed array) to PL
onepllist=[]
for ii in range(len(onepd)):
    oneplsublist = []
    for jj in range(len(onepd[ii])):
        onepddat = np.array(onepd[ii][jj])
        onepdmat = np.transpose( np.resize(onepddat, (3,int(len(onepddat)/3)) ) )
        onepdmat = [onepdmat[onepdmat[:,0]== np.unique(onepdmat[:,0])[h],1:] for h in range(len(np.unique(onepdmat[:,0])))]
        onecircle_pl = PersLandscapeApprox(dgms=onepdmat, hom_deg=1) # compute persistence landscape
        oneplsublist.append(onecircle_pl)
    onepllist.append(oneplsublist)

twopllist=[]
for ii in range(len(twopd)):
    twoplsublist = []
    for jj in range(len(twopd[ii])):
        twopddat = np.array(twopd[ii][jj])
        twopdmat = np.transpose( np.resize(twopddat, (3,int(len(twopddat)/3)) ) )
        twopdmat = [twopdmat[twopdmat[:,0]== np.unique(twopdmat[:,0])[h],1:] for h in range(len(np.unique(twopdmat[:,0])))]
        twocircle_pl = PersLandscapeApprox(dgms=twopdmat, hom_deg=1) # compute persistence landscape
        twoplsublist.append(twocircle_pl)
    twopllist.append(twoplsublist)

In [32]:
random.seed(42)
PL=np.zeros((nsig,nset))
for n in range(nsig):
    print(n)
    for jj in range(nset):
        X = onepllist[n][npc*jj:npc*(jj+1)]
        Y = twopllist[n][npc*jj:npc*(jj+1)]
        PL[n,jj] =permutation_pl_test(X ,Y) # pvalue 

0
1
2
3


In [33]:
savepkl(np.sum(PL<0.05,axis=1)/nset, '../../results/simulation_results/Circle_results/Circles_PL_result_diffnoise.pkl')
print(np.sum(PL<0.05,axis=1)/nset)  

[0.88 0.49 0.25 0.12]
